# LLM Drift Monitor - Colab (Free Tier)

Runs the canary drift-monitoring pipeline against a **free-tier OpenAI-compatible LLM** (e.g. Gemini via Google AI Studio, or Grok via xAI).

## Before you run

1. In Colab: **Runtime -> Change runtime type** -> choose GPU (T4) or keep default CPU - either works.
2. Run the cells below top-to-bottom.
3. In the setup cell, paste your free API key and adjust `OPENAI_BASE_URL` / `OPENAI_MODEL` if needed.

## What this notebook does

- Installs the minimal dependencies.
- Configures the target LLM via environment variables.
- Writes `/content/config.yaml` for the monitor.
- Runs a canary probe batch + monitoring cycle against the live free-tier endpoint.
- Persists results to `/content/drift_monitor.db` (SQLite).

> Drift is a signal for investigation; it does not establish quality degradation without task-specific validation.

In [ ]:
# Install minimal dependencies for the monitoring pipeline
!pip install -q sentence-transformers scipy numpy pyyaml python-dotenv

In [ ]:
import os

# Paste your free-tier API key here (Gemini from Google AI Studio / Grok from xAI)
os.environ['OPENAI_API_KEY'] = 'PASTE_YOUR_FREE_API_KEY_HERE'

# OpenAI-compatible endpoints
os.environ['OPENAI_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'  # Gemini
os.environ['OPENAI_MODEL'] = 'gemini-2.0-flash'  # e.g. 'grok-2' for Grok via https://api.x.ai/v1

print('Using target model:', os.environ['OPENAI_MODEL'])

In [ ]:
%%writefile /content/config.yaml
target:
  type: 'openai-compatible'
  model_id: 'gemini-2.0-flash'
  base_url: 'https://generativelanguage.googleapis.com/v1beta/openai/'
  timeout_seconds: 180
  temperature: 0.0
  max_tokens: 256
embeddings:
  model_name: 'sentence-transformers/all-MiniLM-L6-v2'
  device: 'cpu'
reference:
  window_size: 50
  min_runs: 10
monitoring:
  batch_size: 10
  random_seed: 42
alerts:
  significance_threshold: 0.05
  quality_degradation_threshold: 0.3
  min_sample_size: 10
data:
  db_path: '/content/drift_monitor.db'
dashboard:
  port: 8501
  host: '127.0.0.1'

In [ ]:
import sys

sys.path.insert(0, '/content')

from monitor import LLMDriftMonitor

monitor = LLMDriftMonitor(config_path='/content/config.yaml')
print('Adapter:', type(monitor.adapter).__name__, 'model:', monitor.adapter.model_id)

run_ids = monitor.run_canary_probes(num_prompts=10)
print(f'Inserted {len(run_ids)} probe runs')

result = monitor.run_monitoring_cycle(num_probes=10)
print('Monitoring cycle result:', result)